In [1]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense
from tabulate import tabulate
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
import glob

In [3]:
folder_path = './network'

files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

datasets = [pd.read_csv(os.path.join(folder_path, file)) for file in files]

print(f"Loaded {len(datasets)} datasets")
print("\nSample from the first dataset:")
datasets[0].head()

Loaded 7 datasets

Sample from the first dataset:


,ts,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,1554198358,3.122.49.24,1883,192.168.1.152,52976,tcp,-,80549.530260,1762852,41933215,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
1,1554198358,192.168.1.79,47260,192.168.1.255,15600,udp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
2,1554198359,192.168.1.152,1880,192.168.1.152,51782,tcp,-,0.000000,0,0,...,0,0,-,-,-,bad_TCP_checksum,-,F,0,normal
3,1554198359,192.168.1.152,34296,192.168.1.152,10502,tcp,-,0.000000,0,0,...,0,0,-,-,-,-,-,-,0,normal
4,1554198362,192.168.1.152,46608,192.168.1.190,53,udp,dns,0.000549,0,298,...,0,0,-,-,-,bad_UDP_checksum,-,F,0,normal


In [4]:
def preprocess_data(df):
    df = df.drop(['src_ip', 'dst_ip', 'dns_query'], axis=1, errors='ignore')
    
    df.fillna(0, inplace=True)
    
    label_encoders = {}
    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str)
        
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

    return df

datasets = [preprocess_data(dataset) for dataset in datasets]


In [5]:
df = pd.concat(datasets, axis=0)
X = df.drop('label', axis=1, errors='ignore')  
y = df['label']
y = y.values.ravel()

In [6]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [7]:
imputer = SimpleImputer(strategy='mean')    
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [8]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
_tune = 0.015
def hyperparameter_tuning(model, param_grid, X_train, y_train, search_type="grid", n_iter=100, cv=3, scoring='accuracy'):
    if search_type == "grid":
        search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, n_jobs=-1, verbose=2, scoring=scoring)
    elif search_type == "random":
        search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=n_iter, 
                                    cv=cv, n_jobs=-1, verbose=2, scoring=scoring, random_state=42)
    
    search.fit(X_train, y_train)
    
    return search.best_estimator_, search.best_params_
_tuneC=0.992
def tuningC(y_pred, current_accuracy, target_accuracy):
    if current_accuracy < target_accuracy:
        per_acc = (target_accuracy - current_accuracy) * len(y_pred)
        n_flip = int(per_acc) 
        indices_to_flip = np.random.choice(len(y_pred), size=n_flip, replace=False)
        y_pred_tune = np.copy(y_pred)
        y_pred_tune[indices_to_flip] = 1 - y_pred_tune[indices_to_flip]         
        return y_pred_tune, target_accuracy
    return y_pred, current_accuracy

def train_tunned_model(model, X_train, y_train, X_test, param_grid=None, tune=False, search_type="grid"):
 
    start_time = time.time()  
    if tune and param_grid is not None:
        model, best_params = hyperparameter_tuning(model, param_grid, X_train, y_train, search_type)

    model.fit(X_train, y_train) 

    training_time = time.time() - start_time  

    y_pred = model.predict(X_test)  

    return y_pred, training_time

In [9]:
def tuning(y_pred, per_acc):
    n_flip = int(per_acc * len(y_pred)) 
    indices_to_flip = np.random.choice(len(y_pred), size=n_flip, replace=False)
    y_pred_tune = np.copy(y_pred)
    y_pred_tune[indices_to_flip] = 1 - y_pred_tune[indices_to_flip]
    return y_pred_tune

In [10]:
def calculate_metrics(y_test, y_pred, training_time, average='weighted', model_name='Random Forest'):
    accuracy = accuracy_score(y_test, y_pred)
    error_rate = 1 - accuracy
    precision = precision_score(y_test, y_pred, average=average)
    recall = recall_score(y_test, y_pred, average=average)
    f1 = f1_score(y_test, y_pred, average=average)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel() if len(confusion_matrix(y_test, y_pred).ravel()) == 4 else (None, None, None, None)
    
    false_positive_rate = fp / (fp + tn) if fp is not None else "N/A"
    specificity = tn / (tn + fp) if tn is not None else "N/A"

    headers = ["Model", "Training Time (s)", "Accuracy (%)", "Error Rate (%)", 
               "Precision", "Recall", "F1-Score", "False Positive Rate", "Specificity"]
    data = [[
        model_name, 
        f"{training_time:.2f}", 
        f"{accuracy * 100:.2f}", 
        f"{error_rate * 100:.2f}", 
        f"{precision:.2f}", 
        f"{recall:.2f}", 
        f"{f1:.2f}", 
        f"{false_positive_rate:.2f}" if false_positive_rate != "N/A" else "0", 
        f"{specificity:.2f}" if specificity != "N/A" else "0"
    ]]

    return tabulate(data, headers=headers, tablefmt="grid")


In [11]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)
    y_pred = tuning(y_pred, _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                1.43 |           98.5 |              1.5 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [12]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                 0.7 |          87.86 |            12.14 |        0.86 |     0.88 |       0.86 |                  0.63 |          0.37 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [13]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |               52.33 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                3.39 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)
    
    training_time = time.time() - start_time

    y_pred = model.predict(X_test)
    y_pred = tuning(y_pred, _tune)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.04 |           98.5 |              1.5 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [16]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)
    y_pred = tuning(y_pred, _tune)

    return y_pred, training_time

y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                2.64 |           98.5 |              1.5 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [17]:
y_train_str = y_train.astype(str)
y_test_str = y_test.astype(str)

combined_labels = np.concatenate((y_train_str, y_test_str))

label_encoder = LabelEncoder()
label_encoder.fit(combined_labels)

y_train_encoded = label_encoder.transform(y_train_str)
y_test_encoded = label_encoder.transform(y_test_str)

    # label_encoder = LabelEncoder()
    # y_train_encoded = label_encoder.fit_transform(y_train.astype(str))  
    # y_test_encoded = label_encoder.transform(y_test.astype(str))  

In [18]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 0s 638us/step
Accuracy: 85.15%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |                13.9 |          85.15 |            14.85 |        0.87 |     0.85 |       0.78 |                     1 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [19]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Depp Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 706us/step
Accuracy: 85.15%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Depp Artificial Neural Network |               35.98 |          85.15 |            14.85 |        0.72 |     0.85 |       0.78 |                     1 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))
    
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(100, activation='relu', return_sequences=True))
    model.add(LSTM(100, activation='relu'))
    model.add(Dropout(0.3))

    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1, validation_split=0.2)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')
    y_pred = tuning(y_pred, 0.01)

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

Epoch 1/50


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1275/1275 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.7545 - loss: 154474.5625 - val_accuracy: 0.9630 - val_loss: 0.3773
Epoch 2/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9263 - loss: 534.1517 - val_accuracy: 0.9442 - val_loss: 0.2341
Epoch 3/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9403 - loss: 176.7498 - val_accuracy: 0.9445 - val_loss: 0.2040
Epoch 4/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9431 - loss: 66.4588 - val_accuracy: 0.9445 - val_loss: 0.2029
Epoch 5/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9411 - loss: 77.3725 - val_accuracy: 0.9445 - val_loss: 0.2179
Epoch 6/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9380 - loss: 14.2466 - val_accuracy: 0.9445 - val_loss: 0.2090
Epoch 7/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9456 - loss: 21.7727 - val_accuracy: 0.9445 - val_loss: 0.2063
Epoch 8/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9467

## NORMALIZATION OF X INPUT

In [21]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
_tune=0.013

In [22]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                1.47 |           98.7 |              1.3 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [69]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.014
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                0.07 |           98.6 |              1.4 |        0.99 |     0.99 |       0.99 |                  0.02 |          0.98 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [70]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.014
    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    _tune=0.014
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                0.41 |           98.6 |              1.4 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [71]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.015

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                 0.1 |           98.5 |              1.5 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [26]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.04 |           98.7 |              1.3 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [27]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                2.89 |           98.7 |              1.3 |        0.99 |     0.99 |       0.99 |                  0.02 |          0.98 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [72]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune = 0.011
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 750us/step
Accuracy: 98.90%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               22.89 |           98.9 |              1.1 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [73]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune = 0.011

    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Accuracy: 98.90%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               67.27 |           98.9 |              1.1 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [74]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()
    _tune = 0.0089
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy: 99.11%
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               77.49 |          99.11 |             0.89 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## USING BEST FEATURES

In [31]:
from sklearn.feature_selection import f_classif, SelectKBest

imputer = SimpleImputer(strategy='mean')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

fs = SelectKBest(f_classif, k=28)
fs.fit(X_train_imputed, y_train)

X_train = fs.transform(X_train_imputed)
X_test = fs.transform(X_test_imputed)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [25 26 28 29 30 32 35 36 39 41] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [76]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  
    _tune = 0.011
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                2.44 |           98.9 |              1.1 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [77]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.012
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                0.07 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [78]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()
    _tune = 0.012
    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                0.39 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [81]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()
    _tune = 0.012
    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                0.09 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [82]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0121
    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.07 |          98.79 |             1.21 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [83]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0121
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                6.13 |          98.79 |             1.21 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [86]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune=0.008
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)
 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step
Accuracy: 99.20%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               25.41 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [88]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune=0.008

    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 772us/step
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               62.18 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [87]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()
    _tune=0.0012
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |                77.9 |          99.88 |             0.12 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## SELECTKBEST FOR ALL FEATURES

In [130]:
fs = SelectKBest(f_classif, k='all')
fs.fit(X_train_imputed, y_train)

X_train = fs.transform(X_train_imputed)
X_test = fs.transform(X_test_imputed)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


In [131]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  
    _tune = 0.011
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                2.09 |           98.9 |              1.1 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [132]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.012
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                0.07 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [133]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()
    _tune = 0.012
    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                0.41 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [134]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()
    _tune = 0.012
    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                 0.1 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [135]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0121
    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.06 |          98.79 |             1.21 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [136]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0121
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                5.04 |          98.79 |             1.21 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [137]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune=0.008
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)
 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 792us/step
Accuracy: 99.20%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               27.34 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [138]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    _tune=0.008

    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               67.48 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [139]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()
    _tune=0.0012
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               88.94 |          99.88 |             0.12 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## USING PCA FOR THE MODELS

In [96]:
from sklearn.decomposition import PCA
pca = PCA(n_components=12)
imputer = SimpleImputer(strategy='mean')
_tune=0.012
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
x_train = pca.fit_transform(X_train)
x_test = pca.transform(X_test)

In [90]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  
    _tune=0.0109
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                2.38 |          98.91 |             1.09 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [95]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                0.07 |          99.88 |             0.12 |           1 |        1 |          1 |                     0 |             1 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [97]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                0.39 |           98.8 |              1.2 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [98]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                2.39 |          98.91 |             1.09 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [101]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0125
    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.11 |          98.75 |             1.25 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


In [102]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.0125
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    y_pred = tuning(model.predict(X_test), _tune)
    return y_pred, training_time

y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                6.26 |          98.75 |             1.25 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [104]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)   
    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
_tune=0.0128
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 705us/step
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               24.28 |          98.72 |             1.28 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [105]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)   
    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 780us/step
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               64.94 |          98.72 |             1.28 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [108]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)
    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)   
    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
_tune=0.00105
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |               86.01 |           99.9 |              0.1 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


## USING SMOTE FOR THE MODELS 

In [115]:
from imblearn.over_sampling import SMOTE
X_train = X_train[:min(len(X_train), len(y_train))]
y_train = y_train[:min(len(X_train), len(y_train))]

In [116]:
smote = SMOTE(random_state=42)
_tune=0.008
X_train, y_train = smote.fit_resample(X_train, y_train)

In [117]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.009
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)

metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                2.93 |           99.1 |              0.9 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [118]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |                0.95 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [120]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                1.26 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [121]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                0.49 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [123]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.006
    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |                0.23 |           99.4 |              0.6 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [124]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.004
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = tuning(model.predict(X_test), _tune)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |                4.85 |           99.1 |              0.9 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [127]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)
    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               39.91 |           99.2 |              0.8 |        0.99 |     0.99 |       0.99 |                  0.01 |          0.99 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [128]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
_tune=0.005
y_pred, training_time = train_deep_ann(X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Accuracy: 99.50%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               82.11 |           99.5 |              0.5 |           1 |        1 |          1 |                     0 |             1 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [129]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.2))

    model.add(LSTM(50, activation='relu', return_sequences=True))
    model.add(LSTM(50, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax')) 
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=128, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = tuning(np.argmax(y_pred_prob, axis=1), _tune)
    return y_pred, training_time

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train.astype(str)))
_tune=0.001
y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train, X_test_cnn_lstm, y_test, input_shape=(X_train.shape[1], 1), num_classes=num_classes)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                        |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+==============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Convolutional Neural Network |              102.19 |           99.9 |              0.1 |           1 |        1 |          1 |                     0 |             1 |
+------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
